## Forecast de incendios forestales

El objetivo es proyectar el total de incendios por temporada en la provincia de Colchagua con datos de CONAF. Primero se revisará la cobertura y los valores faltantes; luego se evaluará el pronóstico usando las últimas temporadas y se comparará con referencias simples. Este análisis proyecta incendios, no pérdidas económicas.

In [9]:
from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive", force_remount=True)

ruta_conaf = Path(
    "/content/drive/MyDrive/TRABAJO2 SI/limpios/CONAF_Colchagua_1984_2024_limpio.csv"
)
conaf = pd.read_csv(ruta_conaf, encoding="utf-8-sig")

print("Dimensiones:", conaf.shape)
print("Columnas:", conaf.columns.tolist())
display(conaf.head())

Mounted at /content/drive
Dimensiones: (400, 25)
Columnas: ['region', 'provincia', 'comuna', 'n_incendios', 'sup_pino_0_10', 'sup_pino_11_17', 'sup_pino_18_mas', 'sup_pino_total', 'sup_eucalipto', 'sup_otras_plantaciones', 'sup_plantaciones_total', 'sup_arbolado', 'sup_matorral', 'sup_pastizal', 'sup_vegetacion_natural_total', 'sup_forestal_total', 'sup_agricola', 'sup_desechos', 'sup_otras_superficies_total', 'sup_afectada_total', 'temporada', 'comuna_clave', 'temporada_original', 'anio_inicio', 'anio_fin']


,region,provincia,comuna,n_incendios,sup_pino_0_10,sup_pino_11_17,sup_pino_18_mas,sup_pino_total,sup_eucalipto,sup_otras_plantaciones,...,sup_forestal_total,sup_agricola,sup_desechos,sup_otras_superficies_total,sup_afectada_total,temporada,comuna_clave,temporada_original,anio_inicio,anio_fin
0,VI,COLCHAGUA,Chépica,2.0,NaN,NaN,NaN,0.0,NaN,NaN,...,3.8,NaN,NaN,NaN,3.8,1984-1985,chepica,84-85 oK,1984,1985
1,VI,COLCHAGUA,Chimbarongo,13.0,NaN,NaN,NaN,0.0,NaN,NaN,...,84.9,NaN,NaN,NaN,84.9,1984-1985,chimbarongo,84-85 oK,1984,1985
2,VI,COLCHAGUA,Lolol,7.0,NaN,NaN,NaN,0.0,NaN,NaN,...,67.2,NaN,NaN,NaN,67.2,1984-1985,lolol,84-85 oK,1984,1985
3,VI,COLCHAGUA,Nancagua,5.0,NaN,NaN,NaN,0.0,1.0,NaN,...,43.4,NaN,NaN,NaN,43.4,1984-1985,nancagua,84-85 oK,1984,1985
4,VI,COLCHAGUA,Palmilla,0.0,NaN,NaN,NaN,0.0,NaN,NaN,...,0.0,NaN,NaN,NaN,0.0,1984-1985,palmilla,84-85 oK,1984,1985


### Serie provincial por temporada

Se suman los incendios de las 10 comunas en cada temporada. Los conteos faltantes con superficie afectada igual a cero se consideran cero; los registros completamente vacíos se mantienen como faltantes para no asumir que representan cero incendios.

In [10]:
conaf["n_incendios"] = pd.to_numeric(conaf["n_incendios"], errors="coerce")
conaf["sup_afectada_total"] = pd.to_numeric(
    conaf["sup_afectada_total"], errors="coerce"
)

conaf["incendios_para_forecast"] = conaf["n_incendios"]
casos_cero = (
    conaf["n_incendios"].isna()
    & conaf["sup_afectada_total"].eq(0)
)
conaf.loc[casos_cero, "incendios_para_forecast"] = 0

serie_prov = (
    conaf.groupby(["temporada", "anio_inicio", "anio_fin"], as_index=False)
    .agg(
        comunas=("comuna", "nunique"),
        conteos_disponibles=("incendios_para_forecast", "count"),
        incendios_prov=(
            "incendios_para_forecast",
            lambda x: x.sum(min_count=10)
        )
    )
    .sort_values("anio_inicio")
)

print("Conteos faltantes tratados como cero por superficie afectada igual a 0:", casos_cero.sum())
print("Temporadas con conteo incompleto:", serie_prov.loc[
    serie_prov["incendios_prov"].isna(), "temporada"
].tolist())

display(conaf.loc[
    conaf["incendios_para_forecast"].isna(),
    ["comuna", "temporada", "n_incendios", "sup_afectada_total"]
])

Conteos faltantes tratados como cero por superficie afectada igual a 0: 7
Temporadas con conteo incompleto: ['2015-2016', '2017-2018', '2020-2021']


,comuna,temporada,n_incendios,sup_afectada_total
314,Palmilla,2015-2016,NaN,NaN
334,Palmilla,2017-2018,NaN,NaN
366,Placilla,2020-2021,NaN,NaN


### Evaluación del forecast

Se reserva el tramo final de la serie para prueba. Se compara Holt con dos referencias: repetir el último valor observado y usar el promedio del entrenamiento. Las temporadas con conteos incompletos se excluyen de la evaluación.

In [13]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import Holt
from sklearn.metrics import mean_absolute_error

s = serie_prov.sort_values("anio_inicio").copy()

# Reservar las últimas 8 temporadas para la prueba
inicio_prueba = int(s["anio_inicio"].max()) - 7

train = s[
    (s["anio_inicio"] < inicio_prueba)
    & s["incendios_prov"].notna()
].copy()

test = s[
    (s["anio_inicio"] >= inicio_prueba)
    & s["incendios_prov"].notna()
].copy()

y_train = train["incendios_prov"].astype(float).to_numpy()
y_test = test["incendios_prov"].astype(float).to_numpy()

# Mantener la distancia real entre temporadas
horizontes = (
    test["anio_inicio"].astype(int)
    - int(train["anio_inicio"].max())
).to_numpy()

pred_persistencia = np.repeat(y_train[-1], len(test))
pred_promedio = np.repeat(y_train.mean(), len(test))

modelo_holt = Holt(
    y_train,
    damped_trend=True,
    initialization_method="estimated"
).fit()

pronostico_holt = modelo_holt.forecast(int(horizontes.max()))
pred_holt = pronostico_holt[horizontes - 1]

predicciones = {
    "Persistencia": pred_persistencia,
    "Promedio histórico": pred_promedio,
    "Holt": pred_holt
}

evaluacion = pd.DataFrame([
    {
        "modelo": nombre,
        "MAE": mean_absolute_error(y_test, np.maximum(valores, 0))
    }
    for nombre, valores in predicciones.items()
]).sort_values("MAE")

print("Temporadas completas para prueba:", len(test))
print("Prueba desde:", test["temporada"].min())
display(evaluacion)

Temporadas completas para prueba: 6
Prueba desde: 2016-2017


,modelo,MAE
0,Persistencia,17.333333
1,Promedio histórico,19.182796
2,Holt,28.505726


In [14]:
ultima = (
    serie_prov.dropna(subset=["incendios_prov"])
    .sort_values("anio_inicio")
    .iloc[-1]
)

mae_persistencia = float(
    evaluacion.loc[
        evaluacion["modelo"] == "Persistencia", "MAE"
    ].iloc[0]
)

valor_proyectado = float(ultima["incendios_prov"])
primer_anio = int(ultima["anio_fin"])

proyeccion = pd.DataFrame({
    "temporada": [
        f"{primer_anio + i}-{primer_anio + i + 1}"
        for i in range(3)
    ],
    "incendios_proyectados": [valor_proyectado] * 3,
    "rango_inferior_orientativo": [
        max(0, valor_proyectado - mae_persistencia)
    ] * 3,
    "rango_superior_orientativo": [
        valor_proyectado + mae_persistencia
    ] * 3
})

display(proyeccion.round(1))

,temporada,incendios_proyectados,rango_inferior_orientativo,rango_superior_orientativo
0,2024-2025,80.0,62.7,97.3
1,2025-2026,80.0,62.7,97.3
2,2026-2027,80.0,62.7,97.3


### Interpretación del forecast

La persistencia fue el método con menor error (MAE 17,33) en las 6 temporadas completas de prueba. Proyecta 80 incendios por temporada. El rango de 63 a 97 incendios es orientativo y se basa en el MAE; no es un intervalo de confianza estadístico. La proyección parte de la última temporada disponible, 2023–2024.

In [15]:
from pathlib import Path

salida_forecast = Path("/content/drive/MyDrive/TRABAJO2 SI/ml_forecast")
salida_forecast.mkdir(parents=True, exist_ok=True)

proyeccion.to_csv(
    salida_forecast / "proyeccion_incendios.csv",
    index=False,
    encoding="utf-8-sig"
)
evaluacion.to_csv(
    salida_forecast / "evaluacion_modelos_forecast.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Resultados guardados en:", salida_forecast)

Resultados guardados en: /content/drive/MyDrive/TRABAJO2 SI/ml_forecast
